# Errors

> fastspec errors.

In [ ]:
#| default_exp errors

## Imports

In [ ]:
#| export
from fastcore.utils import *
import json, httpx

In [ ]:
#| hide
from cachy.core import enable_cachy
from fastcore.test import *

In [ ]:
enable_cachy()

## Purpose And Design

`errors.py` defines the error contract used across the entire library.

### `FastSpecError`

Base error.

In [ ]:
#| export
class FastSpecError(Exception):
    "Base fastspec error."

### `SpecError`

Raised when an OpenAPI spec cannot be parsed as expected.

In [ ]:
#| export
class SpecError(FastSpecError):
    "Raised when an OpenAPI spec cannot be parsed as expected."

In [ ]:
#| export
class ProtocolError(FastSpecError):
    "Raised when provider payloads do not match expected protocol shape."

### `_to_text`

Best-effort stringify helper.

In [ ]:
#| export
def _to_text(v):
    "Best-effort stringify helper."
    if v is None: return ""
    if isinstance(v, str): return v
    try: return json.dumps(v, ensure_ascii=False)
    except Exception: return str(v)

### `_retryable`

Classify transient/retryable API failures.

In [ ]:
#| export
def _retryable(status_code, error_type, code, message):
    "Classify transient/retryable API failures."
    if isinstance(status_code, int) and (status_code >= 500 or status_code in (408, 409, 425, 429)): return True
    t = str(error_type or "").lower()
    c = str(code or "").lower()
    m = (message or "").lower()
    hints = ("server_error", "internal", "overload", "rate_limit", "timeout", "unavailable", "temporar")
    if any(h in t for h in hints): return True
    if any(h in c for h in hints): return True
    if any(h in m for h in ("try again", "server error", "temporar", "timeout", "overloaded", "unavailable")): return True
    return False

### `_req_id`

Extract provider request id header when available.

In [ ]:
#| export
def _req_id(headers):
    "Extract provider request id header when available."
    if headers is None: return ""
    for k in ("x-request-id", "request-id", "anthropic-request-id", "x-goog-request-id"):
        v = headers.get(k)
        if v: return str(v)
    return ""

All three major providers (OpenAI, Anthropic, Gemini) consistently return `{"error": <dict>}` with a `message` field inside, so the primary `isinstance(err, dict)` branch handles them all. The remaining branches are defensive fallbacks for edge cases:

- `elif err is not None` — `error` is a string (e.g. proxy/gateway errors)
- `else` — no `error` key at all (e.g. OAuth `error_description`, or `detail`-only responses)
- `except Exception: raw = self.text` — non-JSON responses from CDNs/gateways

Key per-provider differences in the dict branch:
- **[OpenAI](https://platform.openai.com/docs/guides/error-codes)**: `error.{message, type, code, param}`, top-level `status` (int)
- **[Anthropic](https://docs.anthropic.com/en/api/errors)**: `error.{type, message}` only — no `code` or `status`
- **[Gemini](https://ai.google.dev/api/generate-content)**: `error.{code (int), message, status (string), details}` — `status` carries the semantic gRPC code (e.g. `INVALID_ARGUMENT`)

The `status`→`code` promotion at the end ensures semantic codes like Gemini's `"INVALID_ARGUMENT"` are preferred over numeric HTTP codes.

In [ ]:
#| export
@patch
def error(self: httpx.Response):
    "Parse common HTTP API error shapes to (message, error_type, code, raw)."
    try: raw = self.json()
    except Exception: raw = self.text

    msg, et, code, status = "", "", None, None
    if isinstance(raw, dict):
        err = raw.get("error")
        if isinstance(err, dict):
            msg = str(err.get("message") or raw.get("message") or raw.get("detail") or "")
            status = err.get("status", raw.get("status"))
            et = str(err.get("type") or status or raw.get("type") or "")
            code = err.get("code", raw.get("code"))
        elif err is not None:
            msg = _to_text(err)
            status = raw.get("status")
            et = str(raw.get("type") or status or "")
            code = raw.get("code")
        else:
            msg = str(raw.get("message") or raw.get("detail") or raw.get("error_description") or "")
            status = raw.get("status")
            et = str(raw.get("type") or status or "")
            code = raw.get("code")
        if not msg: msg = _to_text(raw)
    else:
        msg = _to_text(raw)
    # Prefer semantic provider codes (e.g. Gemini `status`) over numeric HTTP-like codes.
    if (code is None or isinstance(code, int)) and isinstance(status, str) and status:
        code = status
    if code in (None, "") and isinstance(et, str) and et:
        code = et
    return AttrDict(message=msg, type=et, code=code, raw=raw)

In [ ]:
resp = httpx.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": "Bearer sk-fake123"},
    json={"model": "gpt-4o", "messages": [{"role": "user", "content": "hi"}]}
)
resp.json()

{'error': {'message': 'Incorrect API key provided: sk-fake123. You can find your API key at https://platform.openai.com/account/api-keys.',
  'type': 'invalid_request_error',
  'code': 'invalid_api_key',
  'param': None},
 'status': 401}

In [ ]:
resp.error()

```python
{ 'code': 'invalid_api_key',
  'message': 'Incorrect API key provided: sk-fake123. You can find your API '
             'key at https://platform.openai.com/account/api-keys.',
  'raw': { 'error': { 'code': 'invalid_api_key',
                      'message': 'Incorrect API key provided: sk-fake123. You '
                                 'can find your API key at '
                                 'https://platform.openai.com/account/api-keys.',
                      'param': None,
                      'type': 'invalid_request_error'},
           'status': 401},
  'type': 'invalid_request_error'}
```

In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "hi"}]}
)
resp.json()

{'type': 'error',
 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'},
 'request_id': 'req_011CZawgT2EDaKt1J84fwjww'}

In [ ]:
resp.error()

```python
{ 'code': 'authentication_error',
  'message': 'invalid x-api-key',
  'raw': { 'error': { 'message': 'invalid x-api-key',
                      'type': 'authentication_error'},
           'request_id': 'req_011CZawgT2EDaKt1J84fwjww',
           'type': 'error'},
  'type': 'authentication_error'}
```

In [ ]:
resp = httpx.post(
    "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key=fake123",
    json={"contents": [{"parts": [{"text": "hi"}]}]}
)
resp.json()

{'error': {'code': 400,
  'message': 'API key not valid. Please pass a valid API key.',
  'status': 'INVALID_ARGUMENT',
  'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo',
    'reason': 'API_KEY_INVALID',
    'domain': 'googleapis.com',
    'metadata': {'service': 'generativelanguage.googleapis.com'}},
   {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage',
    'locale': 'en-US',
    'message': 'API key not valid. Please pass a valid API key.'}]}}

In [ ]:
resp.error()

```python
{ 'code': 'INVALID_ARGUMENT',
  'message': 'API key not valid. Please pass a valid API key.',
  'raw': { 'error': { 'code': 400,
                      'details': [ { '@type': 'type.googleapis.com/google.rpc.ErrorInfo',
                                     'domain': 'googleapis.com',
                                     'metadata': { 'service': 'generativelanguage.googleapis.com'},
                                     'reason': 'API_KEY_INVALID'},
                                   { '@type': 'type.googleapis.com/google.rpc.LocalizedMessage',
                                     'locale': 'en-US',
                                     'message': 'API key not valid. Please '
                                                'pass a valid API key.'}],
                      'message': 'API key not valid. Please pass a valid API '
                                 'key.',
                      'status': 'INVALID_ARGUMENT'}},
  'type': 'INVALID_ARGUMENT'}
```

All three providers use different SSE error event shapes:

- **OpenAI** ([docs](https://developers.openai.com/api/reference/resources/responses/streaming-events)): flat `{code, message}` — no nested `error` key, `code` is a string enum (e.g. `"rate_limit_exceeded"`)
- **Anthropic** ([docs](https://docs.anthropic.com/en/api/messages-streaming)): `{type: "error", error: {type, message}}` — no `code` or `status`
- **Gemini** ([docs](https://ai.google.dev/api/generate-content#method:-models.streamgeneratecontent)): `{error: {code, message, status, details}}` — `code` is numeric, `status` is the semantic gRPC code

The extraction logic:
1. `isinstance(event["error"], dict)` → use nested dict (Anthropic, Gemini)
2. `isinstance(event, dict)` but no nested `error` → use whole dict (OpenAI flat shape)
3. Not a dict at all → wrap as `{"message": stringified}` (defensive fallback)

**NB**: Unlike `error()`, this function does *not* promote Gemini's semantic `status` over numeric `code` — so Gemini returns `code=500` rather than `code="INTERNAL"`. This may be worth aligning.

In [ ]:
#| export
def _parse_sse_error_event(event):
    "Parse common SSE `error` event shapes to (message, error_type, code, raw)."
    raw = event
    e = event.get("error") if isinstance(event, dict) and isinstance(event.get("error"), dict) else (
        event if isinstance(event, dict) else {"message": _to_text(event)})
    msg = str(e.get("message") or e.get("detail") or _to_text(e))
    et = str(e.get("type") or e.get("status") or "")
    code = e.get("code")
    if (code is None or isinstance(code, int)) and isinstance(et, str) and et: code = et
    return AttrDict(message=msg, type=et, code=code, raw=raw)

In [ ]:
# OpenAI SSE — flat {code, message}, no nested "error" key
oai_sse = {"code": "rate_limit_exceeded", "message": "Rate limit exceeded"}

# Anthropic SSE — nested error dict with type+message
ant_sse = {"type": "error", "error": {"type": "overloaded_error", "message": "Overloaded"}}

# Gemini SSE — nested error dict with code(int)+message+status(string)
gem_sse = {"error": {"code": 500, "message": "An internal error has occurred", "status": "INTERNAL"}}

for name, evt in [("OpenAI", oai_sse), ("Anthropic", ant_sse), ("Gemini", gem_sse)]:
    error = _parse_sse_error_event(evt)
    print(f"{name:10s} | msg={error.message!r:45s} | et={error.type!r:25s} | code={error.code!r}")

OpenAI     | msg='Rate limit exceeded'                         | et=''                        | code='rate_limit_exceeded'
Anthropic  | msg='Overloaded'                                  | et='overloaded_error'        | code='overloaded_error'
Gemini     | msg='An internal error has occurred'              | et='INTERNAL'                | code='INTERNAL'


### `APIError`

Structured API failure object with optional context.

In [ ]:
#| export
class APIError(FastSpecError):
    "Structured API error with optional context."
    def __init__(self, message: str, *, provider: str = "", model: str = "", endpoint: str = "",
        status_code= None, error_type: str = "", code = None, request_id: str = "",
        retryable = None, raw = None):
        self.message = message or "API request failed"
        self.provider = provider or ""
        self.model = model or ""
        self.endpoint = endpoint or ""
        self.status_code = status_code
        self.error_type = error_type or ""
        self.code = code
        self.request_id = request_id or ""
        self.retryable = _retryable(status_code, self.error_type, self.code, self.message) if retryable is None else bool(retryable)
        self.raw = raw
        super().__init__(self.__str__())

    def with_context(self, *, provider: str = "", model: str = "", endpoint: str = "") -> "APIError":
        "Return a copy with missing context fields filled."
        return APIError(
            self.message,
            provider=self.provider or provider,
            model=self.model or model,
            endpoint=self.endpoint or endpoint,
            status_code=self.status_code,
            error_type=self.error_type,
            code=self.code,
            request_id=self.request_id,
            retryable=self.retryable,
            raw=self.raw,
        )

    def __str__(self):
        fields = ['message', 'provider', 'model', 'endpoint', 'status_code', 'error_type', 'code', 'request_id', 'retryable']
        parts = [f"{f}={getattr(self, f)!r}" for f in fields if getattr(self, f) not in (None, "", False)]
        return f"APIError({', '.join(parts)})"

    __repr__ = __str__

In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "hi"}]}
)
e = resp.error()
err = APIError(e.message, provider="anthropic", model="claude-sonnet-4-20250514",
               endpoint="/v1/messages", status_code=resp.status_code,
               error_type=e.type, code=e.code, request_id=_req_id(resp.headers), raw=e.raw)
err

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='/v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')

In [ ]:
print(err)

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='/v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')


In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "hello!"}]}
)
e = resp.error()
err = APIError(e.message, provider="anthropic", model="claude-sonnet-4-20250514",
               endpoint="/v1/messages", status_code=resp.status_code,
               error_type=e.type, code=e.code, request_id=_req_id(resp.headers), raw=e.raw)
err

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='/v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')

### `api_error_from_http`

Converts `httpx.HTTPStatusError` into `APIError`

In [ ]:
#| export
@patch
def api_error(self:httpx.HTTPStatusError, *, provider: str = "", model: str = "", endpoint: str = ""):
    "Build APIError from httpx HTTPStatusError."
    resp = self.response
    err = resp.error()
    if not endpoint and resp.request is not None:
        endpoint = f"{resp.request.method.upper()} {resp.request.url.path}"
    return APIError(
        err.message,
        provider=provider,
        model=model,
        endpoint=endpoint,
        status_code=resp.status_code,
        error_type=err.type,
        code=err.code,
        request_id=_req_id(resp.headers),
        raw=err.raw,
    )

In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "test api_error_from_http"}]}
)
try: resp.raise_for_status()
except httpx.HTTPStatusError as exc: print(exc.api_error(provider="anthropic", model="claude-sonnet-4-20250514"))

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='POST /v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')


In [ ]:
resp = httpx.get(
    "https://api.github.com/repos/AnswerDotAI/nonexistent-repo-12345",
    headers={"Authorization": "token ghp_fake123"}
)
try: resp.raise_for_status()
except httpx.HTTPStatusError as exc: print(exc.api_error())

APIError(message='Bad credentials', endpoint='GET /repos/AnswerDotAI/nonexistent-repo-12345', status_code=401, error_type='401', code='401')


### `api_error_from_event`

Build APIError from provider SSE/event-level error payload

In [ ]:
#| export
def api_error_from_event(event, *, provider: str = "", model: str = "", endpoint: str = ""):
    "Build APIError from provider SSE/event-level error payload."
    parsed = _parse_sse_error_event(event)
    return APIError(parsed.message, provider=provider, model=model, endpoint=endpoint, error_type=parsed.type, code=parsed.code, raw=parsed.raw)


In [ ]:
for name, evt in [("OpenAI", oai_sse), ("Anthropic", ant_sse), ("Gemini", gem_sse)]:
    err = api_error_from_event(evt, provider=name.lower())
    print(err)

APIError(message='Rate limit exceeded', provider='openai', code='rate_limit_exceeded', retryable=True)
APIError(message='Overloaded', provider='anthropic', error_type='overloaded_error', code='overloaded_error', retryable=True)
APIError(message='An internal error has occurred', provider='gemini', error_type='INTERNAL', code='INTERNAL', retryable=True)


Works with any API — the `provider` and `model` fields are optional context that higher-level libraries can populate:

In [ ]:
# Generic API error — no provider/model needed
gh_evt = {"error": {"message": "API rate limit exceeded", "documentation_url": "https://docs.github.com/rest"}}
print(api_error_from_event(gh_evt, endpoint="GET /repos/{owner}/{repo}"))

APIError(message='API rate limit exceeded', endpoint='GET /repos/{owner}/{repo}')


## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()